# Watem NB01 — Site selection and baseline run
No Google Drive. Select an area, inspect the watershed, run the baseline and download one portable ZIP. Keep that ZIP on your PC for NB02 and NB03. Pilot parameters, not calibrated predictions.

In [ ]:
from pathlib import Path
import os, sys, shutil, subprocess, json, hashlib, time, platform
from datetime import datetime, timezone
ROOT = Path('/content/Watem_local')
DATA = ROOT / 'data/tutorial/langegracht'
OUTPUT = ROOT / 'outputs/student_preparation'
SOFTWARE = ROOT / 'software'
for p in (DATA.parent, OUTPUT, SOFTWARE): p.mkdir(parents=True, exist_ok=True)
REV = '226a7c06ae836f9736c1a88b74b5346033a2a8d6'
REPO = Path('/content/pywatemsedem')
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
WORK = Path('/content/watem_condroz_runs') / RUN_ID
WORK.mkdir(parents=True)
def run(cmd, **kwargs):
    print('+', ' '.join(map(str, cmd)), flush=True)
    return subprocess.run(list(map(str,cmd)), check=True, **kwargs)
print(ROOT, sys.version, platform.machine())
SITE_ID='my_catchment' #@param {type:'string'}
assert SITE_ID and SITE_ID.replace('_','').replace('-','').isalnum()
SITE=ROOT/'student_sites'/f'{SITE_ID}_{RUN_ID}'
assert not (SITE/'baseline'/'complete.json').exists(), 'Site already frozen: use another SITE_ID or go to NB02'
PACKAGE=SITE/'reference'
PACKAGE.mkdir(parents=True,exist_ok=True)
SEED=2271; RES=20; YEAR=2025
# local search window near Ciney; bounds chosen to stay in one DEM tile
SEARCH_LONLAT=(5.00,50.25,5.16,50.36)
TARGET_HA=250; TOL_HA=50
MIN_RIVER_CELLS=20 # >= 20 burned river pixels; not a physical length estimate
# Optional locally cached numerical SPW raster paths; do not use rendered WMS/MapServer imagery.
SPW_DEM=None
SPW_WALOUS=None # requires explicit class crosswalk; not implemented here
run([sys.executable,'-m','pip','install','pysheds','ipyleaflet'])

## 1. Draw area of interest

In [ ]:
from google.colab import output
output.enable_custom_widget_manager()
from ipyleaflet import Map, DrawControl, TileLayer, LayersControl, ScaleControl, FullScreenControl, GeoJSON
import ipywidgets as widgets
from IPython.display import display
selection={}
# Use alternative providers rather than the blocked OSM tile server.
imagery=TileLayer(url='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',attribution='Esri, Maxar, Earthstar Geographics and GIS User Community',name='Aerial / satellite imagery',max_zoom=19)
street=TileLayer(url='https://basemaps.cartocdn.com/rastertiles/voyager/{z}/{x}/{y}.png',attribution='© OpenStreetMap contributors © CARTO',name='Street map',max_zoom=19)
widget=Map(center=(50.36,4.9),zoom=10,layers=(imagery,),scroll_wheel_zoom=True,layout=widgets.Layout(height='520px',width='100%'))
widget.add_control(ScaleControl(position='bottomleft')); widget.add_control(FullScreenControl())
draw=DrawControl(polyline={},circle={},circlemarker={},marker={},polygon={'shapeOptions':{'color':'#ffbb22','fillOpacity':.12}},rectangle={'shapeOptions':{'color':'#ffbb22','fillOpacity':.12}})
status=widgets.HTML('<b>Draw a rectangle or polygon (left toolbar).</b> Zoom to a farming area first.')
from shapely.geometry import shape, mapping, box
overlay=None
def save_geom(g):
    global overlay
    selection['geometry']=g
    if overlay is not None: widget.remove_layer(overlay)
    overlay=GeoJSON(data={'type':'Feature','geometry':g,'properties':{}},style={'color':'#ffbb22','fillOpacity':.08})
    widget.add_layer(overlay)
    b=shape(g).bounds; status.value=f'<b>Area selected.</b> Bounds: {tuple(round(x,4) for x in b)}. Continue to next cell.'
def on_draw(target,action,geo_json):
    if action in ('created','edited'):
        if geo_json.get('type')=='FeatureCollection':
            fs=geo_json.get('features',[])
            if fs: save_geom(fs[-1]['geometry'])
        elif 'geometry' in geo_json: save_geom(geo_json['geometry'])
    elif action=='deleted':
        selection.clear(); status.value='Selection deleted. Draw a new area.'
draw.on_draw(on_draw); widget.add_control(draw)
base=widgets.ToggleButtons(options=['Aerial imagery','Street map'],description='Map:')
def switch(change):
    old=imagery if change['new']=='Street map' else street
    new=street if change['new']=='Street map' else imagery
    if old in widget.layers: widget.substitute_layer(old,new)
base.observe(switch,names='value')
lat=widgets.FloatText(value=50.295,description='Latitude'); lon=widgets.FloatText(value=5.10,description='Longitude'); km=widgets.FloatSlider(value=6,min=2,max=12,step=1,description='Width km')
center=widgets.Button(description='Use centre + width',button_style='info')
def use_center(_):
    import math
    dy=km.value/222; dx=dy/math.cos(math.radians(lat.value))
    save_geom(mapping(box(lon.value-dx,lat.value-dy,lon.value+dx,lat.value+dy)))
    widget.center=(lat.value,lon.value); widget.zoom=12
center.on_click(use_center)
display(widgets.VBox([widgets.HTML('<h3>Choose your study area in Wallonia</h3><p>Draw an approximate area (not the watershed). Alternatively enter a centre and width below. The model will find an eligible DEM-defined catchment.</p>'),base,widget,widgets.HBox([lat,lon]),widgets.HBox([km,center]),status]))


In [ ]:
# Draw a small area on the map above, then run this cell.
from shapely.geometry import shape
assert 'geometry' in selection,'Draw a rectangle or polygon first'
AOI=shape(selection['geometry']); SEARCH_LONLAT=AOI.bounds
assert AOI.is_valid and AOI.area>0
assert 2.8<SEARCH_LONLAT[0]<SEARCH_LONLAT[2]<6.5 and 49.4<SEARCH_LONLAT[1]<SEARCH_LONLAT[3]<51,'Select a small area within Wallonia (approximate bounds)'
assert SEARCH_LONLAT[2]-SEARCH_LONLAT[0]<.25 and SEARCH_LONLAT[3]-SEARCH_LONLAT[1]<.20,'Draw a smaller AOI (roughly 5–15 km across)'
print(SEARCH_LONLAT)


## 2. Install and prepare model

In [ ]:
import urllib.request
os_release = dict(line.strip().split('=',1) for line in Path('/etc/os-release').read_text().splitlines() if '=' in line)
codename = os_release.get('VERSION_CODENAME','').strip('"')
assert platform.machine() == 'x86_64', 'Expected x86_64 Colab runtime'
assert codename in {'jammy','noble','focal','bookworm'}, f'Unsupported OS for pinned SAGA release: {codename}'
run(['apt-get','update','-qq'])
run(['apt-get','install','-y','git-lfs','saga'])
run(['git','lfs','install'])
def download(url, path):
    if not path.exists():
        tmp = path.with_suffix(path.suffix+'.part')
        urllib.request.urlretrieve(url, tmp)
        tmp.replace(path)
    return path
saga_name = f'saga-vpo_2025.6.11-1-ge95f136.{codename}_amd64.deb'
saga_deb = download('https://github.com/DOV-Vlaanderen/saga-watem/releases/download/2025.6.11/'+saga_name, SOFTWARE/saga_name)
local_deb = Path('/tmp') / saga_name
shutil.copy2(saga_deb, local_deb)
run(['apt-get','install','-y',str(local_deb)])
ws = download('https://github.com/watem-sedem/watem-sedem/releases/download/5.0.3/watem_sedem', SOFTWARE/'watem_sedem_5.0.3')
binary = Path('/usr/local/bin/watem_sedem')
shutil.copy2(ws,binary); binary.chmod(0o755)
os.environ['WATEMSEDEM'] = str(binary.parent)
# saga-vpo contains plugins, not the SAGA command-line program.
# Ubuntu's saga package supplies saga_cmd.
saga_cmd = shutil.which('saga_cmd')
if not saga_cmd:
    run(['dpkg','-L','saga'])
    raise RuntimeError('saga_cmd missing after installing saga')
assert shutil.which('watem_sedem'), 'watem_sedem missing from PATH'
print('SAGA executable:', saga_cmd)
print('WaTEM executable:', shutil.which('watem_sedem'))
plugin_files = subprocess.check_output(['dpkg','-L','saga-vpo'],text=True).splitlines()
print('Installed SAGA plugin libraries:', [p for p in plugin_files if p.endswith('.so')])
run(['saga_cmd','--version'])
for args in (['saga_cmd','topology'], ['saga_cmd','watem']):
    p = subprocess.run(args,capture_output=True,text=True)
    msg = p.stdout+p.stderr
    print(msg[:3500])
    assert 'error: select a library' not in msg.lower() and 'cannot load' not in msg.lower(), f'Missing/incompatible SAGA library: {args}'
run(['ldd',str(binary)])
if not REPO.exists():
    run(['git','clone','https://github.com/watem-sedem/pywatemsedem.git',REPO],
        env={**os.environ,'GIT_LFS_SKIP_SMUDGE':'1'})
run(['git','-C',REPO,'checkout',REV],env={**os.environ,'GIT_LFS_SKIP_SMUDGE':'1'})
run([sys.executable,'-m','pip','install',str(REPO)])
import pywatemsedem
print('pywatemsedem',pywatemsedem.__version__)

In [ ]:
required = ['dtm.tif','K3.tif','basemap_landuse.tif','userchoices.ini']
stems = ['catchm_langegracht','river','infrastructure','roads','water','parcels','grass_strips','buffers','force_routing','outlets']
source = REPO/'tests/data'
names = sorted(p.name for p in source.iterdir() if p.is_file() and (p.name in required or p.stem in stems))
assert set(required).issubset(names)
for stem in stems:
    assert all(stem+ext in names for ext in ('.shp','.shx','.dbf','.prj')), stem
def expected_hash(name):
    raw = subprocess.check_output(['git','-C',str(REPO),'show',f'{REV}:tests/data/{name}'])
    if raw.startswith(b'version https://git-lfs'):
        return next(x.split(':',1)[1] for x in raw.decode().splitlines() if x.startswith('oid sha256:'))
    return hashlib.sha256(raw).hexdigest()
hashes = {name:expected_hash(name) for name in names}
import urllib.request, urllib.error
from concurrent.futures import ThreadPoolExecutor, as_completed

def valid_file(path, name):
    return path.is_file() and hashlib.sha256(path.read_bytes()).hexdigest() == hashes[name]

def valid(folder):
    return all(valid_file(folder/n,n) for n in names)

DATA.mkdir(parents=True,exist_ok=True)
missing = [n for n in names if not valid_file(DATA/n,n)]
print(f'Valid cached files: {len(names)-len(missing)}/{len(names)}')
if missing:
    # Git LFS pull can fail at a detached HEAD or due to upstream LFS service.
    # Capture stderr in the notebook, rather than only showing CalledProcessError.
    pending = [n for n in missing if not valid_file(source/n,n)]
    if pending:
        p = subprocess.run(['git','-C',str(REPO),'lfs','pull','origin',
                            '--include='+','.join('tests/data/'+n for n in pending)],
                           capture_output=True,text=True)
        print('Git LFS return code:',p.returncode)
        print(p.stdout[-4000:])
        print(p.stderr[-6000:])
    def retrieve(name):
        if valid_file(DATA/name,name):
            return name,'cached'
        if valid_file(source/name,name):
            shutil.copy2(source/name,DATA/name)
            return name,'git-lfs'
        # GitHub media endpoint serves actual LFS objects rather than pointers.
        # Always require the exact SHA256 from the pinned upstream Git tree.
        errors=[]
        tmp=DATA/(name+'.part')
        for base in (
            f'https://media.githubusercontent.com/media/watem-sedem/pywatemsedem/{REV}/tests/data/',
            f'https://github.com/watem-sedem/pywatemsedem/raw/{REV}/tests/data/',
            f'https://raw.githubusercontent.com/watem-sedem/pywatemsedem/{REV}/tests/data/',
        ):
            for attempt in range(2):
                try:
                    req=urllib.request.Request(base+name,headers={'User-Agent':'Watem-Colab/1.0'})
                    with urllib.request.urlopen(req,timeout=60) as response, tmp.open('wb') as dst:
                        shutil.copyfileobj(response,dst)
                    if not valid_file(tmp,name):
                        raise ValueError('SHA256 mismatch (possibly a Git LFS pointer)')
                    tmp.replace(DATA/name)
                    return name,'direct download'
                except Exception as e:
                    errors.append(f'{base}: {type(e).__name__}: {e}')
                    tmp.unlink(missing_ok=True)
                    if attempt==0: time.sleep(2)
        raise RuntimeError(name+': '+' | '.join(errors))
    failures=[]
    with ThreadPoolExecutor(max_workers=4) as pool:
        futures={pool.submit(retrieve,name):name for name in missing}
        for f in as_completed(futures):
            try:
                print(*f.result())
            except Exception as e:
                failures.append(str(e))
                print('FAILED:',e)
    if failures:
        raise RuntimeError(f'{len(failures)} data downloads failed. Valid files are retained on Drive. See HTTP/LFS diagnostics above.')
assert valid(DATA), 'Tutorial data checksum validation failed'
(DATA/'source_manifest.json').write_text(json.dumps({'revision':REV,'sha256':hashes},indent=2))
inputdata_folder = WORK/'inputs'
inputdata_folder.mkdir(exist_ok=True)
for n in names: shutil.copy2(DATA/n,inputdata_folder/n)
print(f'Validated {len(names)} files in {DATA}')
print('Local working data:', inputdata_folder)

## 3. Terrain, stream burning, D8 watershed

In [ ]:
import numpy as np, pandas as pd, geopandas as gpd, rasterio, requests, matplotlib.pyplot as plt
from rasterio.warp import reproject, Resampling, transform_bounds
from rasterio.transform import from_origin, xy
from rasterio.features import shapes, rasterize
from shapely.geometry import shape, Point, box
from pysheds.grid import Grid
from IPython.display import display
CRS='EPSG:31370'
bb=transform_bounds('EPSG:4326',CRS,*SEARCH_LONLAT,densify_pts=21)
xmin,ymin=np.floor(np.array(bb[:2])/RES)*RES; xmax,ymax=np.ceil(np.array(bb[2:])/RES)*RES
# buffer around candidate domain; still within tile except possibly west edge; mosaic both adjacent tiles
xmin-=2000; ymin-=2000; xmax+=2000; ymax+=2000
width,height=int((xmax-xmin)/RES),int((ymax-ymin)/RES)
tr=from_origin(xmin,ymax,RES,RES)
profile=dict(driver='GTiff',height=height,width=width,count=1,crs=CRS,transform=tr,nodata=-9999,compress='deflate')
def warp_sources(urls,method):
    dest=np.full((height,width),-9999,dtype='float32')
    for url in urls:
        tmp=np.full_like(dest,-9999)
        with rasterio.open(url) as ds:
            reproject(rasterio.band(ds,1),tmp,dst_transform=tr,dst_crs=CRS,dst_nodata=-9999,resampling=method)
        dest[tmp!=-9999]=tmp[tmp!=-9999]
    assert np.isfinite(dest).all() and (dest!=-9999).all(),'Incomplete raster coverage'
    return dest
def dem_url(lon,lat):
    tile=f'Copernicus_DSM_COG_10_N{lat:02d}_00_E{lon:03d}_00_DEM'
    return f'https://copernicus-dem-30m.s3.amazonaws.com/{tile}/{tile}.tif'
import math
west,south,east,north=transform_bounds(CRS,'EPSG:4326',xmin,ymin,xmax,ymax)
dem_sources=[str(SPW_DEM)] if SPW_DEM else [dem_url(lon,lat) for lon in range(math.floor(west),math.floor(east)+1) for lat in range(math.floor(south),math.floor(north)+1)]
wc_url=[f'https://esa-worldcover.s3.eu-central-1.amazonaws.com/v200/2021/map/ESA_WorldCover_10m_2021_v200_N{lat:02d}E{lon:03d}_Map.tif' for lat in range(math.floor(south/3)*3,math.floor(north/3)*3+1,3) for lon in range(math.floor(west/3)*3,math.floor(east/3)*3+1,3)]
dem=warp_sources(dem_sources,Resampling.bilinear)
lc=warp_sources(wc_url,Resampling.nearest).astype('int16')
def write_raster(path,a,dtype='float32',transform=tr):
    p=profile.copy(); p.update(dtype=dtype,height=a.shape[0],width=a.shape[1],transform=transform)
    with rasterio.open(path,'w',**p) as ds: ds.write(a.astype(dtype),1)
write_raster(WORK/'search_dem.tif',dem)
print('Grid:',dem.shape,'Elevation range:',dem.min(),dem.max(),'Land cover:',np.unique(lc))

In [ ]:
# Retrieve the river network across the FULL buffered DEM region before delineation.
BASE='https://geoservices.wallonie.be/arcgis/rest/services'
sources={'parcels':f'{BASE}/AGRICULTURE/SIGEC_PARC_AGRI_{YEAR}/MapServer/4','soils':f'{BASE}/SOL_SOUS_SOL/CNSW/MapServer/2','roads':f'{BASE}/TOPOGRAPHIE/PICC_VDIFF/MapServer/21','buildings':f'{BASE}/TOPOGRAPHIE/PICC_VDIFF/MapServer/11','river':f'{BASE}/TOPOGRAPHIE/PICC_VDIFF/MapServer/28','water':f'{BASE}/TOPOGRAPHIE/PICC_VDIFF/MapServer/30'}
session=requests.Session()
def request_json(url,params):
    response=session.get(url,params=params,timeout=120); response.raise_for_status()
    j=response.json()
    if 'error' in j: raise RuntimeError(j['error'])
    return j
def fetch_features(url, aoi=None):
    aoi=boundary if aoi is None else aoi
    meta=request_json(url,{'f':'json'})
    assert meta.get('geometryType'),f'Not a feature layer: {url}'
    bounds=aoi.total_bounds
    spatial=dict(where='1=1',geometry=','.join(map(str,bounds)),geometryType='esriGeometryEnvelope',inSR=31370,spatialRel='esriSpatialRelIntersects',returnIdsOnly='true',f='json')
    ids=request_json(url+'/query',spatial).get('objectIds') or []
    features=[]
    for i in range(0,len(ids),100):
        batch=request_json(url+'/query',dict(objectIds=','.join(map(str,ids[i:i+100])),outFields='*',outSR=4326,returnGeometry='true',f='geojson'))
        assert not batch.get('exceededTransferLimit'),'Truncated batch'
        features.extend(batch['features'])
    assert len(features)==len(ids),(len(features),len(ids))
    if not features: return gpd.GeoDataFrame(geometry=[],crs=CRS),meta
    g=gpd.GeoDataFrame.from_features(features,crs=4326).to_crs(CRS)
    g.geometry=g.geometry.make_valid()
    return gpd.clip(g,aoi,keep_geom_type=True),meta

search_aoi=gpd.GeoDataFrame(geometry=[box(xmin,ymin,xmax,ymax)],crs=CRS)
river_search,river_meta=fetch_features(sources['river'],search_aoi)
assert river_meta['geometryType']=='esriGeometryPolyline' and len(river_search)>0
river_search=river_search.explode(index_parts=False).reset_index(drop=True)
river_search=river_search[river_search.geometry.geom_type=='LineString'].copy()
assert len(river_search)>0
river_search.to_file(WORK/'river_search.gpkg',driver='GPKG')
print('River features in buffered search area:',len(river_search))
# SAGA native grids via GDAL conversion; stream background must be NoData, not zero.
# Method 0 avoids inventing an authoritative direction from an unoriented vector network.
# A post-fill D8 check then requires mapped river connectivity within selected basin.
BURN_DEPTH=1.0
MIN_SLOPE=0.01
dem_original=dem.copy()
river_cells=rasterize([(g,1) for g in river_search.geometry],out_shape=dem.shape,transform=tr,fill=0,all_touched=True,dtype='uint8').astype(bool)
write_raster(WORK/'stream_mask.tif',np.where(river_cells,1,-9999),'int16')
def saga(*args):
    cmd=['saga_cmd',*map(str,args)]
    proc=subprocess.run(cmd,capture_output=True,text=True,errors='replace')
    print(' '.join(cmd),proc.returncode,flush=True)
    if proc.returncode: print(proc.stdout[-4000:],proc.stderr[-4000:]); proc.check_returncode()
def to_saga(src,dst):
    # Import with SAGA's GDAL tool (preserves NoData and grid geometry).
    saga('io_gdal',0,'-FILES',src,'-GRIDS',dst)
    assert Path(dst).exists(),dst
def from_saga(src,dst):
    saga('io_gdal',2,'-GRIDS',src,'-FILE',dst)
    with rasterio.open(dst) as ds:
        assert ds.shape==dem.shape and ds.transform.almost_equals(tr), (ds.shape,ds.transform,tr)
        return ds.read(1).astype('float32')
to_saga(WORK/'search_dem.tif',WORK/'original.sgrd')
to_saga(WORK/'stream_mask.tif',WORK/'streams.sgrd')
saga('ta_preprocessor',6,'-DEM',WORK/'original.sgrd','-STREAM',WORK/'streams.sgrd','-BURN',WORK/'burned.sgrd','-METHOD',0,'-EPSILON',BURN_DEPTH)
saga('ta_preprocessor',4,'-ELEV',WORK/'burned.sgrd','-FILLED',WORK/'hydro.sgrd','-FDIR',WORK/'hydro_fdir.sgrd','-WSHED',WORK/'hydro_wshed.sgrd','-MINSLOPE',MIN_SLOPE)
dem_burned=from_saga(WORK/'burned.sgrd',WORK/'search_dem_burned.tif')
dem_hydro=from_saga(WORK/'hydro.sgrd',WORK/'search_dem_hydro_export.tif')
assert np.isfinite(dem_hydro).all() and (dem_hydro!=-9999).all()
write_raster(WORK/'search_dem_hydro.tif',dem_hydro)
# Independent steepest D8 slope check, using pysheds default D8 ordering.
# Interior cells only: boundary cells may legitimately drain outside search domain.
dirs=[(-1,0),(-1,1),(0,1),(1,1),(1,0),(1,-1),(0,-1),(-1,-1)]
best=np.full(dem.shape,-np.inf); river_downhill=np.zeros(dem.shape,dtype=bool)
for dr,dc in dirs:
    nbr=np.roll(dem_hydro,(-dr,-dc),axis=(0,1))
    nriv=np.roll(river_cells,(-dr,-dc),axis=(0,1))
    slope=(dem_hydro-nbr)/(RES*np.hypot(dr,dc))
    better=slope>best
    river_downhill[better]=nriv[better] & (slope[better]>0)
    best[better]=slope[better]
interior=np.ones(dem.shape,dtype=bool); interior[[0,-1],:]=False; interior[:,[0,-1]]=False
pits=interior & (best<=0)
assert not pits.any(),f'{pits.sum()} interior cells lack a downhill neighbour'
bad_river=river_cells & interior & ~river_downhill
print('Interior pits/flats:',int(pits.sum()),'River cells without downhill mapped D8 successor (includes natural endpoints):',int(bad_river.sum()))
delta=dem_hydro-dem_original
print('Elevation modification (m): min/median/max',np.min(delta),np.median(delta),np.max(delta))
fig,axes=plt.subplots(1,3,figsize=(16,6),constrained_layout=True)
for ax,a,title in zip(axes,[dem_original,dem_hydro,delta],['Original DEM','Hydrologically conditioned DEM','Elevation modification (m)']):
    im=ax.imshow(a,cmap='terrain' if title!='Elevation modification (m)' else 'RdBu_r')
    ax.contour(river_cells,levels=[.5],colors='blue',linewidths=.3)
    ax.set_title(title); fig.colorbar(im,ax=ax,shrink=.65)
fig.savefig(WORK/'hydro_conditioning.png',dpi=180); plt.show()


In [ ]:
# Compatibility with NumPy >= 2.4: pysheds still uses the removed np.in1d.
# Preserve the old one-dimensional return shape and signature.
if not hasattr(np, 'in1d'):
    def _in1d(ar1, ar2, assume_unique=False, invert=False):
        return np.isin(np.asarray(ar1).ravel(), np.asarray(ar2).ravel(),
                       assume_unique=assume_unique, invert=invert)
    np.in1d = _in1d
    print('Applied pysheds / NumPy in1d compatibility shim.')
grid=Grid.from_raster(str(WORK/'search_dem_hydro.tif'))
z=grid.read_raster(str(WORK/'search_dem_hydro.tif'))
fdir=grid.flowdir(z)
acc=grid.accumulation(fdir)
acc_a=np.asarray(acc)
# candidate outlet within original search bbox (not the surrounding buffer)
rr,cc=np.indices(dem.shape)
xx=xmin+(cc+.5)*RES; yy=ymax-(rr+.5)*RES
candidate=np.argwhere(river_cells & (acc_a*RES**2/10000>=TARGET_HA-TOL_HA)&(acc_a*RES**2/10000<=TARGET_HA+TOL_HA)&(xx>=bb[0])&(xx<=bb[2])&(yy>=bb[1])&(yy<=bb[3]))
from shapely import contains_xy
local_aoi=gpd.GeoSeries([AOI],crs=4326).to_crs(CRS).iloc[0]
keep=contains_xy(local_aoi,xx[candidate[:,0],candidate[:,1]],yy[candidate[:,0],candidate[:,1]])
candidate=candidate[keep]
rng=np.random.default_rng(SEED); rng.shuffle(candidate)
print('Candidate river outlets in drawn AOI:',len(candidate))
chosen=None
for row,col in candidate[:500]:
    m=np.asarray(grid.catchment(x=int(col),y=int(row),fdir=fdir,xytype='index'),dtype=bool)
    if not m.any() or m[0,:].any() or m[-1,:].any() or m[:,0].any() or m[:,-1].any(): continue
    area=m.sum()*RES**2/10000
    if abs(area-TARGET_HA)>TOL_HA or np.mean(np.isin(lc[m],[30,40]))<.5: continue
    # Every interior basin river cell must follow a lower mapped river cell, except outlet.
    bad=m & river_cells & ~river_downhill
    bad[row,col]=False
    if bad.any() or int((m & river_cells).sum())<MIN_RIVER_CELLS: continue
    chosen=(int(row),int(col),m); break
assert chosen is not None,'No acceptable river-connected basin: inspect hydro QA, enlarge search region or tolerance; do not silently relax drainage checks.'
row,col,mask=chosen
# Confirm exact D8 closure: every non-outlet pixel has one downstream cell within basin.
D8_OFFSETS={64:(-1,0),128:(-1,1),1:(0,1),2:(1,1),4:(1,0),8:(1,-1),16:(0,-1),32:(-1,-1)}
fd=np.asarray(fdir); rix,cix=np.where(mask); bad=[]
for rr0,cc0 in zip(rix,cix):
    if (rr0,cc0)==(row,col): continue
    off=D8_OFFSETS.get(int(fd[rr0,cc0]))
    if off is None: bad.append((rr0,cc0)); continue
    rt,ct=rr0+off[0],cc0+off[1]
    if not mask[rt,ct] or dem_hydro[rt,ct]>=dem_hydro[rr0,cc0]: bad.append((rr0,cc0))
assert not bad, f'D8 basin has {len(bad)} internal drainage inconsistencies'
assert int(acc_a[row,col])==int(mask.sum()), 'D8 accumulation and basin cell count differ'
print('D8 closure PASS; burned river cells:',int((mask & river_cells).sum()))
outxy=xy(tr,row,col)
boundary=gpd.GeoDataFrame(geometry=[shape(g) for g,v in shapes(mask.astype('uint8'),mask=mask,transform=tr) if v==1],crs=CRS).dissolve()
outlet=gpd.GeoDataFrame(geometry=[Point(*outxy)],crs=CRS)
print('Chosen area ha:',mask.sum()*RES**2/10000,'outlet lon/lat:',outlet.to_crs(4326).geometry.iloc[0])
boundary.to_file(PACKAGE/'catchment.shp'); outlet.to_file(PACKAGE/'outlet.gpkg',driver='GPKG')
fig,ax=plt.subplots(figsize=(9,8)); boundary.plot(ax=ax,facecolor='#e6e5d8',edgecolor='black'); gpd.clip(river_search,boundary).plot(ax=ax,color='blue',linewidth=2); outlet.plot(ax=ax,color='red',markersize=40); ax.set_title('Selected DEM-derived basin | mapped streams and outlet'); plt.show()
print(f'Agriculture/grass: {100*np.mean(np.isin(lc[mask],[30,40])):.1f}% | D8 area: {mask.sum()*RES**2/10000:.2f} ha')
from ipyleaflet import GeoJSON
bgeo=json.loads(boundary.to_crs(4326).to_json()); ogeo=json.loads(outlet.to_crs(4326).to_json())
widget.add_layer(GeoJSON(data=bgeo,style={'color':'#00e5ff','fillOpacity':.1},name='Delineated watershed'))
widget.add_layer(GeoJSON(data=ogeo,name='Outlet')); display(widget)

## 4. Accept the delineated basin
Inspect the preceding map. Set the checkbox true only if suitable; otherwise change the seed or AOI and rerun from a fresh runtime.

In [ ]:
ACCEPT=False #@param {type:'boolean'}
assert ACCEPT,'Inspect the watershed and tick ACCEPT before continuing'

## 5. Download data and prepare native model

In [ ]:
vectors={}; metadata={}
for name,url in sources.items():
    print('Fetching',name,flush=True)
    if name=='river':
        vectors[name]=gpd.clip(river_search,boundary,keep_geom_type=True)
        metadata[name]=river_meta
    else:
        vectors[name],metadata[name]=fetch_features(url)
    expected={'parcels':'esriGeometryPolygon','soils':'esriGeometryPolygon','roads':'esriGeometryPolyline','buildings':'esriGeometryPolygon','river':'esriGeometryPolyline','water':'esriGeometryPolygon'}[name]
    assert metadata[name]['geometryType']==expected,(name,metadata[name]['geometryType'])
    if name=='water' and 'NATUR_DESC' in vectors[name]:
        vectors[name]=vectors[name][vectors[name].NATUR_DESC!='Piscine'].copy()
    print(name,len(vectors[name]),list(vectors[name].columns))
    if len(vectors[name]): vectors[name].to_file(PACKAGE/f'{name}_source.gpkg',driver='GPKG')
    (PACKAGE/f'{name}_schema.json').write_text(json.dumps(metadata[name],indent=2))
assert len(vectors['parcels']) and len(vectors['soils'])
display(vectors['parcels'].drop(columns='geometry').head())
display(vectors['soils'].drop(columns='geometry').head())

In [ ]:
p=vectors['parcels'].copy()
fields=metadata['parcels'].get('fields',[])
crop_fields=[f['name'] for f in fields if any(k in (f.get('name','')+' '+f.get('alias','')).lower() for k in ['culture','crop'])]
print('Candidate crop fields:',crop_fields)
# User can set CROP_FIELD explicitly after viewing source schema.
CROP_FIELD='CULT_NOM'
assert CROP_FIELD in p.columns,'Identify official crop attribute in schema'
# Decode coded value domain where provided
field=next(f for f in fields if f['name']==CROP_FIELD)
domain={str(x['code']):x['name'] for x in (field.get('domain') or {}).get('codedValues',[])}
p['crop_label']=p[CROP_FIELD].astype(str).map(lambda x:domain.get(x,x))
def crop_params(label):
    x=label.lower()
    for keys,c,cn,cl in [(['prairie','herbe','grass'],.01,61,'grass'),(['maïs','mais','maize'],.61,81,'maize'),(['pomme de terre','potato'],.64,81,'potato'),(['betterave','beet'],.41,81,'beet'),(['colza','rapeseed'],.38,81,'rape'),(['froment','orge','épeautre','avoine','seigle','triticale','céréale','cereal'],.39,76,'cereal'),(['lin','chanvre'],.68,81,'flax_hemp')]:
        if any(k in x for k in keys): return c,cn,cl
    return .37,81,'UNKNOWN'
p[['C_crop','CN_base','crop_group']]=pd.DataFrame([crop_params(x) for x in p.crop_label],index=p.index)
p['LANDUSE']=np.where(p.crop_group=='grass',-4,-9999).astype(int)
p['C_reduct']=0.0
p['parcel_id']=np.arange(1,len(p)+1)
p['area_ha']=p.geometry.area/10000
display(p.groupby(['crop_label','crop_group','C_crop','CN_base'],dropna=False).area_ha.agg(['count','sum']).reset_index())
p.drop(columns='geometry').to_csv(PACKAGE/'parcel_parameters.csv',index=False)
p.to_file(PACKAGE/'parcels.gpkg',driver='GPKG')
p[['LANDUSE','C_crop','C_reduct','parcel_id','geometry']].to_file(PACKAGE/'parcels.shp')
if (p.crop_group=='UNKNOWN').any(): print('WARNING: unknown crop labels use provisional C/CN; inspect CSV before scientific use.')
# Crop and management tables for future student interface
pd.DataFrame([('baseline',1,0),('CN_only',1,-10),('C_only',.6,0),('combined',.6,-10)],columns=['scenario','C_multiplier','CN_delta']).to_csv(PACKAGE/'management_scenarios.csv',index=False)

In [ ]:
# crop domain subset with 2-cell margin, retaining grid alignment
r0,r1=np.where(mask)[0].min()-10,np.where(mask)[0].max()+11
c0,c1=np.where(mask)[1].min()-10,np.where(mask)[1].max()+11
sl=np.s_[r0:r1,c0:c1]; subtr=tr*rasterio.Affine.translation(c0,r0); m=mask[sl]; cover=lc[sl]
# WaTEM landuse codes: 10 generic crop, -3 forest, -4 pasture, -5 pond, -2 infrastructure
bg=np.full(m.shape,10,dtype='int16')
for k,v in {10:-3,20:-4,30:-4,40:10,50:-2,60:10,70:10,80:-5,90:-4,95:-4,100:-4}.items(): bg[cover==k]=v
bg[~m]=-9999
write_raster(PACKAGE/'dtm_original.tif',dem_original[sl],transform=subtr)
write_raster(PACKAGE/'dtm.tif',dem_hydro[sl],transform=subtr)
write_raster(PACKAGE/'dtm_difference.tif',(dem_hydro-dem_original)[sl],transform=subtr)
write_raster(PACKAGE/'basemap_landuse.tif',bg,'int16',subtr)
write_raster(PACKAGE/'K3.tif',np.where(m,30,-9999),'int16',subtr)
write_raster(PACKAGE/'HSG_provisional.tif',np.where(m,2,-9999),'int16',subtr)
for name in ['roads','buildings','river','water']:
    g=vectors[name]
    if len(g):
        g=g[['geometry']].copy()
        if name=='roads': g['paved']=-2
        g.to_file(PACKAGE/(('infrastructure' if name=='buildings' else name)+'.shp'))
fig,ax=plt.subplots(2,3,figsize=(15,9),constrained_layout=True)
for a,data,title in [(ax[0,0],dem_hydro[sl],'Conditioned elevation (m)'),(ax[0,1],bg,'Background land cover'),(ax[0,2],np.where(m,30,np.nan),'Provisional K')]:
    im=a.imshow(np.where(m,data,np.nan)); a.set_title(title); fig.colorbar(im,ax=a,shrink=.7)
p.plot(ax=ax[1,0],column='C_crop',legend=True,vmin=0,vmax=1); ax[1,0].set_title('Parcel C factor')
vectors['soils'].plot(ax=ax[1,1],facecolor='none',edgecolor='black'); ax[1,1].set_title('Actual CNSW soil polygons')
boundary.boundary.plot(ax=ax[1,2],color='black')
for name,color in [('roads','grey'),('river','blue'),('buildings','red')]:
    if len(vectors[name]): vectors[name].plot(ax=ax[1,2],color=color)
ax[1,2].set_title('PICC infrastructure and drainage')
fig.savefig(PACKAGE/'input_atlas.png',dpi=180); plt.show()
manifest=dict(seed=SEED,year=YEAR,area_ha=float(m.sum()*RES**2/10000),outlet_lonlat=list(outlet.to_crs(4326).geometry.iloc[0].coords)[0],resolution=RES,dem=dem_sources,landcover=wc_url,vector_services=sources,K_provisional=30,HSG_provisional='B',unknown_crop_count=int((p.crop_group=='UNKNOWN').sum()),hydro_conditioning={'burn_depth_m':BURN_DEPTH,'min_slope_deg':MIN_SLOPE,'river_source':sources['river'],'method':'SAGA burn method 0 then Wang-Liu fill; strict downstream mapped river D8 check'},scientific_status='pilot assumptions; not calibrated')
(PACKAGE/'metadata.json').write_text(json.dumps(manifest,indent=2))
print('Saved prepared inputs:',PACKAGE)

In [ ]:
from pywatemsedem.catchment import Catchment
from pywatemsedem.choices import Choices, Options, Parameters, Extensions, ExtensionsParameters, Output
from pywatemsedem.scenario import Scenario
import configparser
assert (PACKAGE/'river.shp').exists(),'No mapped river in selected basin; choose another seed or implement verified flow-path network.'
catchment=Catchment(SITE_ID,PACKAGE/'catchment.shp',PACKAGE/'dtm.tif',RES,31370,-9999,results_folder=WORK)
catchment.kfactor=PACKAGE/'K3.tif'
catchment.landuse=PACKAGE/'basemap_landuse.tif'
catchment.vct_river=PACKAGE/'river.shp'
for prop,name in [('vct_infrastructure_buildings','infrastructure'),('vct_infrastructure_roads','roads'),('vct_water','water')]:
    path=PACKAGE/f'{name}.shp'
    if path.exists(): setattr(catchment,prop,path)
default_choices=inputdata_folder/'userchoices.ini'
def settings():
    o=Options(); o.read_values_from_ini(default_choices)
    par=Parameters(); par.read_values_from_ini(default_choices)
    ext=Extensions(); ext.read_values_from_ini(default_choices)
    out=Output(); out.read_values_from_ini(default_choices)
    ep=ExtensionsParameters(ext); ep.apply_defaults()
    for v in (ep.ktc_low,ep.ktc_high,ep.ktc_limit): v.read_value_from_ini(default_choices)
    return Choices(o,par,ext,ep,out)
rain=np.array([0,2,4,7,9,8,5,3,2,0],dtype=float)
rainfile=WORK/'storm.txt'; np.savetxt(rainfile,np.column_stack([np.arange(len(rain))*10,rain]),fmt=['%d','%.3f'],delimiter='\t')
intensity=rain[1:]*6; R=float(np.sum(.29*(1-.72*np.exp(-.05*intensity))*rain[1:])*np.max(np.convolve(rain[1:],np.ones(3),'valid'))*2)
scenarios={}
for nr,mode in enumerate(['annual','event'],1):
    s=Scenario(catchment,YEAR,nr,settings())
    if mode=='event':
        s.choices.extensions.curve_number=True; s.choices.extensions.convert_output=False
        s.choices.parameters.r_factor=R
        ep=s.choices.extensionparameters; ep.alpha=0.; ep.beta=0.; ep.antecedent_rainfall=0.; ep.stream_velocity=.2; ep.desired_timestep=60; ep.endtime_model=720
        s.choices.output.write_total_runoff=True; s.choices.output.write_rainfall_excess=True
    s.vct_parcels=PACKAGE/'parcels.shp'
    s.composite_landuse=s.create_composite_landuse(); s.cfactor=s.create_cfactor()
    ep=s.choices.extensionparameters
    s.ktc=s.create_ktc(ep.ktc_low.value,ep.ktc_high.value,ep.ktc_limit.value,not s.choices.extensions.create_ktc_map.value)
    if mode=='event':
        lu=np.asarray(s.composite_landuse.arr); inside=np.asarray(catchment.mask.arr)==1
        cn=np.zeros(lu.shape,dtype='float32')
        cn[inside & (lu>0)]=81
        # Use model grid profile, not the source raster shape.
        pr=s.rp.rasterio_profile
        crop_cn=rasterize([(g,float(v)) for g,v in zip(p.geometry,p.CN_base)],out_shape=lu.shape,transform=pr['transform'],fill=0,dtype='float32')
        use=inside & (lu>0)&(crop_cn>0); cn[use]=crop_cn[use]
        for k,v in {-1:100,-2:98,-3:55,-4:61,-5:100,-6:61}.items(): cn[inside & (lu==k)]=v
        assert np.all((cn[inside]>0)&(cn[inside]<=100)),np.unique(lu[inside & (cn==0)])
        s.cn=cn; s.rainfall=rainfile
    # Explicitly refill the entire wrapper rectangular DEM from the full conditioned search grid.
    from rasterio.warp import reproject, Resampling
    model_profile=s.rp.rasterio_profile
    full=np.full(s.composite_landuse.arr.shape,np.nan,dtype='float32')
    reproject(dem_hydro,full,src_transform=tr,src_crs=CRS,dst_transform=model_profile['transform'],dst_crs=CRS,dst_nodata=np.nan,resampling=Resampling.nearest)
    assert np.isfinite(full).all() and (full>-1000).all(), 'Model rectangular window lacks valid DEM coverage'
    catchment.dtm.arr=full.copy()
    s.prepare_input_files(); s.create_ini_file()
    # Verify actual exported DTM, PRC and river pixels before running.
    native_dir=s.sfolder.wsinput_folder
    ini_check=configparser.ConfigParser(interpolation=None); ini_check.read(s.ini)
    dtm_candidates=[v for sec in ini_check.sections() for k,v in ini_check[sec].items() if 'dtm' in k.lower() and v.lower().endswith(('.rst','.tif'))]
    assert len(dtm_candidates)==1,dtm_candidates
    dtm_path=native_dir/Path(dtm_candidates[0]).name
    with rasterio.open(dtm_path) as ds:
        before=ds.read(1,masked=True)
        native_transform=ds.transform
        native_crs=ds.crs
        print('Wrapper DEM before repair:', 'nodata',ds.nodata,'masked',int(np.ma.getmaskarray(before).sum()),'raw range',float(np.min(before.data)),float(np.max(before.data)))
        assert ds.shape==full.shape and native_transform.almost_equals(model_profile['transform']), 'Native grid alignment mismatch'
    # Write directly using GDAL RST CreateCopy via rasterio, including a new RDC.
    # Verify the round trip and grid alignment before replacing native files.
    from rasterio.shutil import copy as rio_copy
    tif=WORK/f'{mode}_full_dtm.tif'
    with rasterio.open(tif,'w',driver='GTiff',height=full.shape[0],width=full.shape[1],count=1,dtype='float32',crs=CRS,transform=model_profile['transform'],nodata=-99999.) as ds:
        ds.write(full,1)
    temp=WORK/f'{mode}_full_dtm.rst'
    rio_copy(tif,temp,driver='RST')
    with rasterio.open(temp) as ds:
        raw=ds.read(1,masked=True)
        assert ds.dtypes[0]=='float32' and not np.ma.getmaskarray(raw).any() and np.allclose(raw.data,full,atol=.001), 'Direct RST creation failed'
        assert ds.transform.almost_equals(native_transform)
    # Replace both raster and header; delete stale GDAL auxiliary metadata.
    for suffix in ['.rst','.rdc']:
        shutil.copy2(temp.with_suffix(suffix),dtm_path.with_suffix(suffix))
    for suffix in ['.aux.xml','.msk','.ovr']:
        Path(str(dtm_path)+suffix).unlink(missing_ok=True)
    with rasterio.open(dtm_path) as ds:
        native_dem=ds.read(1,masked=True)
        print('Repaired native DEM:',ds.shape,ds.dtypes[0],'nodata',ds.nodata,'masked',int(np.ma.getmaskarray(native_dem).sum()),'range',float(native_dem.data.min()),float(native_dem.data.max()))
        assert ds.transform.almost_equals(native_transform) and ds.shape==full.shape
        assert not np.ma.getmaskarray(native_dem).any() and np.isfinite(native_dem.data).all() and (native_dem.data>-1000).all(), 'Native DEM contains invalid elevations'
        assert np.allclose(native_dem.data,full,atol=.001), 'Native DEM differs from full window'
    lu=np.asarray(s.composite_landuse.arr); inside=np.asarray(catchment.mask.arr)==1
    print(mode,'model cells',inside.sum(),'river cells',np.sum(inside & (lu==-1)),'DEM range',float(full.min()),float(full.max()))
    assert np.any(inside & (lu==-1)), 'No native river cells in basin'
    print('Outside-domain parcel codes:',np.unique(lu[~inside]))
    # Native executable recognizes only PRC=0 as outside; -9999 is NOT an outside code.
    ini_files={k:v for sec in ini_check.sections() for k,v in ini_check[sec].items() if v.lower().endswith('.rst')}
    print('INI raster references:',ini_files)
    def native_read(key):
        matches=[v for k,v in ini_files.items() if key(k)]
        assert len(matches)==1,matches
        with rasterio.open(native_dir/Path(matches[0]).name) as ds: return ds.read(1),ds.transform
    prc_matches=[v for k,v in ini_files.items() if 'parcel' in k.lower() and 'filename' in k.lower()]
    if len(prc_matches)!=1:
        prc_matches=[f.name for f in native_dir.glob('*.rst') if 'parcelmosaic' in f.stem.lower() or f.stem.lower() in ('prc','perceelskaart')]
    assert len(prc_matches)==1,('Cannot locate executable parcel raster',ini_files)
    prc_path=native_dir/Path(prc_matches[0]).name
    with rasterio.open(prc_path) as ds: native_prc=ds.read(1); prc_transform=ds.transform
    print('Native PRC outside raw codes BEFORE:',np.unique(native_prc[~inside]),'inside codes:',np.unique(native_prc[inside]))
    assert prc_transform.almost_equals(native_transform)
    # Rebuild native PRC from actual model array and explicit outside=0, not NoData.
    prc=np.where(inside,lu,0).astype('int16')
    assert np.all((prc[inside]>0)|(np.isin(prc[inside],[-1,-2,-3,-4,-5,-6])))
    assert np.max(prc)<=32757,'Parcel IDs exceed model range'
    prctif=WORK/f'{mode}_prc.tif'; prcrst=WORK/f'{mode}_prc.rst'
    with rasterio.open(prctif,'w',driver='GTiff',height=prc.shape[0],width=prc.shape[1],count=1,dtype='int16',crs=CRS,transform=native_transform,nodata=-9999) as ds: ds.write(prc,1)
    rio_copy(prctif,prcrst,driver='RST')
    for suffix in ['.rst','.rdc']: shutil.copy2(prcrst.with_suffix(suffix),prc_path.with_suffix(suffix))
    for suffix in ['.aux.xml','.msk','.ovr']: Path(str(prc_path)+suffix).unlink(missing_ok=True)
    with rasterio.open(prc_path) as ds:
        native_prc=ds.read(1); assert np.array_equal(native_prc,prc)
    print('Native PRC outside raw codes AFTER:',np.unique(native_prc[~inside]),'native river cells:',int((native_prc==-1).sum()))
    assert np.all(native_prc[~inside]==0) and (native_prc==-1).sum()>=10,'Native stream representation too sparse'
    # Constant provisional K is defined on the entire model grid, not masked by
    # a separately rasterized watershed polygon. PRC alone defines active cells.
    k_matches=[v for k,v in ini_files.items() if k.lower()=='k factor filename']
    assert len(k_matches)==1,k_matches
    kpath=native_dir/Path(k_matches[0]).name
    with rasterio.open(kpath) as ds:
        k_before=ds.read(1)
        assert ds.shape==full.shape and ds.transform.almost_equals(native_transform)
    invalid_k=inside & (~np.isfinite(k_before) | (k_before<0))
    print(mode,'invalid K before repair:',int(invalid_k.sum()),'ha:',float(invalid_k.sum()*RES**2/10000))
    # Quantify the original D8 mask versus wrapper mask before making repairs.
    d8_grid=np.zeros(inside.shape,dtype='uint8')
    reproject(mask.astype('uint8'),d8_grid,src_transform=tr,src_crs=CRS,dst_transform=native_transform,dst_crs=CRS,dst_nodata=0,resampling=Resampling.nearest)
    print('D8 vs wrapper mask: D8 cells',int(d8_grid.sum()),'wrapper cells',int(inside.sum()),'wrapper-only',int((inside & (d8_grid==0)).sum()),'D8-only',int(((d8_grid==1)&~inside).sum()))
    fig,ax=plt.subplots(figsize=(8,7)); ax.imshow(full,cmap='terrain'); ax.contour(inside,levels=[.5],colors='black',linewidths=.7); ax.contour(d8_grid,levels=[.5],colors='cyan',linewidths=.7); rrk,cck=np.where(invalid_k); ax.scatter(cck,rrk,s=9,c='red',label=f'{len(rrk)} invalid K cells'); ax.legend(); ax.set_title(mode+' | K input mismatch (black wrapper, cyan D8)'); plt.show()
    k_fixed=np.full(full.shape,30,dtype='int16')
    ktif=WORK/f'{mode}_k_full.tif'; krst=WORK/f'{mode}_k_full.rst'
    with rasterio.open(ktif,'w',driver='GTiff',height=full.shape[0],width=full.shape[1],count=1,dtype='int16',crs=CRS,transform=native_transform,nodata=-9999) as ds: ds.write(k_fixed,1)
    rio_copy(ktif,krst,driver='RST')
    for suffix in ['.rst','.rdc']: shutil.copy2(krst.with_suffix(suffix),kpath.with_suffix(suffix))
    for suffix in ['.aux.xml','.msk','.ovr']: Path(str(kpath)+suffix).unlink(missing_ok=True)
    with rasterio.open(kpath) as ds:
        k_after=ds.read(1,masked=True)
        assert ds.transform.almost_equals(native_transform) and ds.dtypes[0]=='int16'
        assert not np.ma.getmaskarray(k_after).any() and np.array_equal(k_after.data,k_fixed)
    print(mode,'K repair: all grid cells = 30 kg h/(MJ mm); invalid active cells = 0')
    # Verify raw numerical parameter values in the active model domain.
    for k,v in ini_files.items():
        if not any(x in k.lower() for x in ['c factor','c-factor','k factor','k-factor','p factor','p-factor','ktc']): continue
        with rasterio.open(native_dir/Path(v).name) as ds:
            a=ds.read(1); vv=a[inside & (native_prc!=-1)]
            print(k,'active min/max',float(vv.min()),float(vv.max()),'negative cells',int((vv<0).sum()))
            assert np.isfinite(vv).all() and (vv>=0).all(),f'Invalid active {k} values'
            if 'p factor' in k.lower() or 'p-factor' in k.lower(): assert (vv>0).all(),'P=0 triggers invalid RUSLE sentinel in Pascal 5.0.3'
    fig,ax=plt.subplots(figsize=(8,7)); ax.imshow(np.ma.masked_where(~inside,full),cmap='terrain'); ax.contour(inside,levels=[.5],colors='black',linewidths=.7); rrv,ccv=np.where(native_prc==-1); ax.scatter(ccv,rrv,s=12,color='blue',label=f'{len(rrv)} native river pixels'); ax.legend(); ax.set_title(mode+' | actual native PRC river cells'); plt.show()
    np.savez_compressed(WORK/f'{mode}_native_domain.npz',prc=native_prc,mask=inside,dem=full)
    assert not np.any(lu[~inside]>0), 'Agricultural parcel IDs outside model domain'
    if mode=='event':
        cfg=configparser.ConfigParser(interpolation=None); cfg.optionxform=str; cfg.read(s.ini)
        sec=next(x for x in cfg.sections() if x.lower()=='parameters extensions')
        for key,value in [('Endtime model',720),('Bulk density',int(s.choices.parameters.bulk_density.value))]:
            actual=next((x for x in cfg[sec] if x.lower()==key.lower()),key); cfg[sec][actual]=str(value)
        with s.ini.open('w') as f: cfg.write(f)
    scenarios[mode]=s
    print(mode,s.ini)
DEST=OUTPUT/RUN_ID; DEST.mkdir(exist_ok=True)
for mode,s in scenarios.items(): shutil.copytree(s.sfolder.scenario_folder,DEST/mode,dirs_exist_ok=True)
print('Native inputs copied; INI paths refer to current local runtime. Regenerate on next runtime.')

## 6. Run and report baseline

In [ ]:
# Run baseline annual and 40 mm event using the verified native inputs.
import urllib.request, sys
helper_dir=Path('/content/watem_student_helpers'); helper_dir.mkdir(exist_ok=True)
urllib.request.urlretrieve('https://raw.githubusercontent.com/kvanoost/Watem/50839661b3a72d2f7a78b7c3d0a4ba7019908cc7/watem_student_common.py',helper_dir/'watem_student_common.py')
sys.path.insert(0,str(helper_dir))
import watem_student_common as ws

BASELINE=SITE/'baseline'; BASELINE.mkdir(exist_ok=True)
rows=[]
for mode,s in scenarios.items():
    dest=BASELINE/('annual' if mode=='annual' else 'event_40mm')
    ws.clone_run(s.sfolder.scenario_folder,dest,binary,40 if mode=='event' else None)
    ws.execute(dest,binary)
    rows.append(ws.report(dest,dest.name,40 if mode=='event' else None))
# Two additional storms, same duration and shape.
for depth in (20,60):
    dest=BASELINE/f'event_{depth}mm'
    ws.clone_run(BASELINE/'event_40mm',dest,binary,depth); ws.execute(dest,binary)
    rows.append(ws.report(dest,dest.name,depth))
display(pd.DataFrame(rows)); pd.DataFrame(rows).to_csv(BASELINE/'summary.csv',index=False)
(BASELINE/'complete.json').write_text(json.dumps(dict(kind='baseline',baseline_id=ws.baseline_digest(BASELINE),site=SITE_ID,year=YEAR,resolution=RES,storms=[20,40,60],parameter_status='provisional K=30, HSG B, annual C used for event',reference=str(PACKAGE)),indent=2))


## 7. Download GIS package

In [ ]:
# Empty shapefile templates with explicit schema, readable in ArcGIS and QGIS.
import fiona
from google.colab import files
GIS=WORK/'GIS_baseline'; GIS.mkdir(exist_ok=True)
for sub in ['reference','baseline_results','editable_scenario']: (GIS/sub).mkdir()
for f in PACKAGE.iterdir():
    if f.is_file() and f.suffix.lower() in ['.shp','.shx','.dbf','.prj','.cpg','.gpkg','.tif','.csv','.json']: shutil.copy2(f,GIS/'reference'/f.name)
for mode in ['annual','event_20mm','event_40mm','event_60mm']: shutil.copytree(BASELINE/mode/'report',GIS/'baseline_results'/mode)
schemas={'C_change':('Polygon',{'id':'int','c_mult':'float','practice':'str:80','years':'int','comment':'str:200'}),'CN_change':('Polygon',{'id':'int','cn_delta':'float','soc_base':'float','soc_new':'float','years':'int','practice':'str:80','comment':'str:200'}),'grass':('LineString',{'id':'int','width_m':'float','type':'str:40','comment':'str:200'})}
for name,(geom,props) in schemas.items():
    with fiona.open(GIS/'editable_scenario'/f'{name}.shp','w',driver='ESRI Shapefile',crs='EPSG:31370',schema={'geometry':geom,'properties':props}): pass
(GIS/'README.txt').write_text('''Open layers in ArcGIS Pro or QGIS. Edit ONLY the three shapefile templates. Draw polygons for C multiplier (0–1) and CN delta (-20 to 0); draw grass centerlines with total width in metres. Save edits; ZIP the editable_scenario folder including SHP, SHX, DBF, PRJ. Upload in NB02. SOC fields document assumptions only. Parameters are provisional; compare relative effects, not validated predictions. River outlet discharge is the model-reported automatic outlet. Annual C is reused for events in this pilot.''')
# A single portable ZIP contains GIS layers and machine-readable native baseline inputs.
shutil.copytree(BASELINE,GIS/'baseline',dirs_exist_ok=True)
archive=shutil.make_archive(str(SITE/f'{SITE_ID}_Baseline'),'zip',GIS)
print('Save this ZIP locally: it is needed for NB02 and NB03.'); files.download(archive)
